# Shopee Modular Data Crawler

Pipeline chia làm 3 bước độc lập để đảm bảo **tính toàn vẹn dữ liệu** và **khả năng chạy tiếp (resume)** khi gặp lỗi.

**Lưu ý quan trọng trước khi chạy:**
1. Tắt hết trình duyệt Chrome hiện tại.
2. Mở cmd/terminal và chạy lệnh sau để mở Chrome chế độ Remote Debugging:
   ```bash
   "C:\Program Files\Google\Chrome\Application\chrome.exe" --remote-debugging-port=9222 --user-data-dir="C:\chrome_debug_profile"
   ```
3. Trên cửa sổ Chrome mới mở ra, hãy truy cập Shopee.vn và **Đăng nhập tài khoản** bằng tay.
4. Sau khi đăng nhập xong, bạn có thể chạy tuần tự các ô bên dưới.

## Bước 1: Thu thập danh sách Sản phẩm (Products)

Crawl sản phẩm theo từng sub-category, lưu file CSV riêng cho mỗi sub-category.
Sau khi xong 1 category → merge thành file tổng hợp. Xong tất cả → merge thành `products.csv`.

**Checkpoint:** Nếu bị gián đoạn, chạy lại cell này sẽ tự động resume từ sub-category/page đang dở.

**Tham số `max_pages`:**
- `max_pages=1` → Demo nhanh (1 trang/sub-category)
- `max_pages=5` → Mặc định (~300 SP/sub-category)
- `max_pages=None` hoặc số lớn → Lấy hết

In [ ]:
from crawl_products import crawl_products_pipeline

products_df = crawl_products_pipeline()

if products_df is not None:
    print(f"\nTotal products: {len(products_df)}")
    display(products_df.head(10))

Connecting to Chrome...
Connected to Chrome successfully!

 CRAWL PRODUCTS (with checkpoint)

  CATEGORY: Thời trang Nam

  Thời trang Nam > Áo Khoác
    Page 1... [8 pages from UI] 60 new / 60 items (total: 60)
    Page 2/8... 

## Bước 2: Bổ sung thông tin Shop (Shops)

Đọc `products.csv` → lấy danh sách `shop_id` duy nhất → gọi API shop.
Có tính năng **Resume**: bỏ qua shop đã cào.

In [ ]:
from crawl_shops import crawl_shops_pipeline

shops_df = crawl_shops_pipeline(
    products_file="shopee_data/products.csv", 
    output_file="shopee_data/shops.csv"
)

if shops_df is not None:
    print(f"\nTotal shops: {len(shops_df)}")
    display(shops_df.head())

 Skip 5567 existing shops from previous crawl.
Connecting to Chrome...
Connected to Chrome successfully!

 CRAWL SHOP DETAILS
 Cần crawl mới: 71 shop
   [1/71] Shop ID: 1225189834 
   [2/71] Shop ID: 19420247 
   [3/71] Shop ID: 148885922 
   [4/71] Shop ID: 1457910130 
   [5/71] Shop ID: 500024302 
   [6/71] Shop ID: 60758316 
   [7/71] Shop ID: 964474189 
   [8/71] Shop ID: 164773207 
   [9/71] Shop ID: 834756665 
   [10/71] Shop ID: 983530276 
   --- Resting 48s after 10 shops to avoid block ---
   [11/71] Shop ID: 98124463 
   [12/71] Shop ID: 364520176 
   [13/71] Shop ID: 365335862 
   [14/71] Shop ID: 1376737846 
   [15/71] Shop ID: 1374126569 
   [16/71] Shop ID: 696833266 
   [17/71] Shop ID: 1058270532 
   [18/71] Shop ID: 947032478 
   [19/71] Shop ID: 8639445 
   [20/71] Shop ID: 914773221 
   --- Resting 58s after 20 shops to avoid block ---
   [21/71] Shop ID: 1263615028 
   [22/71] Shop ID: 28233984 
   [23/71] Shop ID: 403471147 
   [24/71] Shop ID: 833394873 
   [25/71

,shop_id,shop_name,location,rating_star,follower_count,is_official_shop,item_count,response_rate,response_time,crawled_at
0,962733632,Sthgruybfub.vn,NaN,4.784238,33775,False,9879,100,799,2026-03-07T21:51:00.837947
1,1008809970,Cà vạt chuyên dụng yzq007.vn,Nước ngoài,4.833333,1879,False,96,100,6497,2026-03-07T21:51:13.492509
2,322530470,ewann.vn,Nước ngoài,4.904741,11968,True,4429,97,6513,2026-03-07T21:51:26.016777
3,183199642,Binstore thời trang nam,Hà Tĩnh,4.829123,12683,False,142,91,2361,2026-03-07T21:51:38.013949
4,308456542,cheesenhujm.vn,NaN,4.666000,14692,False,3412,100,767,2026-03-07T21:51:50.661447


## Bước 3: Thu thập Bình luận (Reviews/Ratings)

Đọc `products.csv` → vào từng sản phẩm lấy review.
Có tính năng **Resume**: bỏ qua product đã lấy review.

In [2]:
from crawl_reviews import crawl_reviews_pipeline

reviews_df = crawl_reviews_pipeline(
    products_file="shopee_data/products.csv", 
    output_file="shopee_data/reviews.csv"
)

if reviews_df is not None:
    print(f"\nTotal reviews: {len(reviews_df)}")
    display(reviews_df.head())

Connecting to Chrome...
Connected to Chrome successfully!

 CRAWL REVIEWS
 Total: 1498 | Done: 0 | Remaining: 1498
 Reviews per product: All

  RESUMING from checkpoint:
  Last product: 22475246212 | Page: 2
  Completed: 0/1498
   Resuming product 22475246212 from page 2 (18 existing reviews)
   [1/1498] Á *NEW* Dép Nam Nữ Quai Ngang Thời ... (resume p.2) 
      [p.3] 6 ratings, +0 new, total=18, has_more=True -> dup (1/3), skipping ahead

      [p.4] 6 ratings, +6 new, total=24, has_more=True
      [p.5] 6 ratings, +6 new, total=30, has_more=True
      [p.6] 3 ratings, +3 new, total=33, has_more=False -> DONE
(33/33 reviews, p.6/6)
   [2/1498] [Xả kho hình ảnh minh họa] Bộ Đồ Na... 
      [p.1] 6 ratings, +6 new, total=6, has_more=True
      [p.2] 6 ratings, +6 new, total=12, has_more=True
      [p.3] 6 ratings, +6 new, total=18, has_more=True
      [p.4] 6 ratings, +6 new, total=24, has_more=True
      [p.5] 6 ratings, +6 new, total=30, has_more=True
      [p.6] 6 ratings, +6 new, to

KeyboardInterrupt: 